# 09 — End-to-End Pipeline: Load → Trim → Analyze → Export

This notebook assembles every concept introduced in notebooks 01–08 into a
single, reproducible analysis pipeline.  By the end you will have:

- Loaded a three-member ensemble
- Trimmed each member with the **strategy-operation** pattern
- Computed ensemble statistics using all three aggregation techniques
- Exported the best estimate
- Compared three different trimming strategies side by side

The pipeline structure is identical regardless of the number of members or the
trimming strategy used — only the configuration objects change.

---

## Pipeline Overview

```
CSV files
   │
   ▼ Step 1 — Load
List[DataStream]  (raw, untrimmed)
   │
   ▼ Step 2 — Explore
trace_plot, is_stationary
   │
   ▼ Step 3 — Trim  (strategy-operation pattern)
List[DataStream]  (trimmed, steady-state only)
   │
   ▼ Step 4 — Ensemble
Ensemble object
   │
   ▼ Step 5 — Visualize steady state
   │
   ▼ Step 6 — Statistics (techniques 0, 1, 2)
Dict of results
   │
   ▼ Step 7 — Export
DataFrame / JSON
```

---

## Step 1 — Load Raw Ensemble Members

In [ ]:
import quends as qnds

files = [
    "gx/ensemble/tprim_2_5_a.out.csv",
    "gx/ensemble/tprim_2_5_b.out.csv",
    "gx/ensemble/tprim_2_5_c.out.csv",
]

members = [qnds.from_csv(f) for f in files]

print(f"Loaded {len(members)} ensemble members")
for i, m in enumerate(members):
    print(f"  Member {i}: {len(m)} steps,  t_max = {m.data['time'].max():.1f}")

---

## Step 2 — Explore Raw Data

Before trimming, inspect the first member visually and check stationarity.
The trace plot shows the transient growth phase and helps you decide whether
the automatic trim boundary looks sensible.

In [ ]:
plotter = qnds.Plotter()

# Raw trace — always the first diagnostic
plotter.trace_plot(members[0], ["HeatFlux_st"])

In [ ]:
# Stationarity test — returns True if the signal appears stationary
is_stat = members[0].is_stationary("HeatFlux_st")
print(f"Member 0 (raw) is stationary: {is_stat}")

---

## Step 3 — Trim Using the Strategy-Operation Pattern

Each `DataStream` is trimmed individually.  Members that emerge empty
(i.e. the algorithm found no steady-state window) are dropped before
building the ensemble.

**Rule:** always use `QuantileTrimStrategy` + `TrimDataStreamOperation`.
Never call `ds.trim()`.

In [ ]:
from quends.base.trim import QuantileTrimStrategy, TrimDataStreamOperation

strategy = QuantileTrimStrategy(window_size=20, robust=True)
op = TrimDataStreamOperation(strategy=strategy)

# Apply to every member
trimmed_members = [op(m, column_name="HeatFlux_st") for m in members]

# Drop members with empty steady-state windows
trimmed_members = [t for t in trimmed_members if not t.data.empty]

print(f"{len(trimmed_members)} of {len(members)} members survived trimming")
for i, t in enumerate(trimmed_members):
    print(f"  Member {i}: SS starts at t = {t.data['time'].iloc[0]:.2f}")

---

## Step 4 — Build the Trimmed Ensemble

In [ ]:
ens = qnds.Ensemble(trimmed_members)
print(f"Ensemble ready: {len(ens.data_streams)} members")

---

## Step 5 — Visualize the Steady-State Window

`steady_state_automatic_plot` overlays the detected trim point on the raw
trace, giving an intuitive view of what the algorithm selected.

In [ ]:
plotter.steady_state_automatic_plot(members[0], ["HeatFlux_st"])

---

## Step 6 — Compute Ensemble Statistics (All Techniques)

Run all three aggregation techniques and collect results for comparison.

| Technique | Strategy |
|-----------|----------|
| `0` | Concatenate all members, then estimate once |
| `1` | Average per-member estimates |
| `2` | Hierarchical pooled estimator (**recommended** for UQ) |

In [ ]:
results = {}

for t in [0, 1, 2]:
    r = ens.compute_statistics("HeatFlux_st", technique=t)
    results[f"T{t}"] = r
    print(f"Technique {t}: mean = {r['mean']:.4f},  CI = {r['confidence_interval']}")

---

## Step 7 — Export the Best Result

Technique 2 (hierarchical pooled) is the recommended estimator for ensemble
UQ because it correctly propagates both within-member and between-member
variance.

In [ ]:
exporter = qnds.Exporter()

# DataFrame view (ideal for Jupyter)
exporter.display_dataframe(results["T2"])

In [ ]:
# JSON view (ideal for downstream scripts / archiving)
exporter.display_json(results["T2"])

---

## Applying a Different Trimming Strategy

The pipeline is strategy-agnostic: swap in a different `*TrimStrategy` object
without touching anything else.  Here we use `IQRTrimStrategy`, which
detects steady state via the interquartile range of a rolling window.

In [ ]:
from quends.base.trim import IQRTrimStrategy, TrimDataStreamOperation

strategy_iqr = IQRTrimStrategy(window_size=20, threshold=0.05)
op_iqr = TrimDataStreamOperation(strategy=strategy_iqr)

trimmed_iqr = [op_iqr(m, column_name="HeatFlux_st") for m in members]
trimmed_iqr = [t for t in trimmed_iqr if not t.data.empty]

ens_iqr = qnds.Ensemble(trimmed_iqr)
r_iqr = ens_iqr.compute_statistics("HeatFlux_st", technique=2)

print(f"IQR strategy: mean = {r_iqr['mean']:.4f}")

---

## Comparing Multiple Trimming Strategies

Use `build_trim_strategy()` as a factory to iterate over strategy names without
importing each class individually.  This is the recommended pattern for
sensitivity studies on the trim method.

In [ ]:
from quends.base.trim import build_trim_strategy, TrimDataStreamOperation

print(f"{'Method':<22} {'mean':>10}  {'n_members':>10}")
print("-" * 46)

for method in ["std", "iqr", "rolling_variance"]:
    strategy = build_trim_strategy(method=method, window_size=20)
    op = TrimDataStreamOperation(strategy=strategy)

    survived = [op(m, column_name="HeatFlux_st") for m in members]
    survived = [t for t in survived if not t.data.empty]

    if survived:
        ens_m = qnds.Ensemble(survived)
        r = ens_m.compute_statistics("HeatFlux_st", technique=2)
        print(f"{method:<22} {r['mean']:>10.4f}  {len(survived):>10}")
    else:
        print(f"{method:<22} {'N/A':>10}  {0:>10}")

---

## Summary

This notebook demonstrated the complete `quends` analysis pipeline:

| Step | Key API |
|------|---------|
| **Load** | `qnds.from_csv(path)` |
| **Explore** | `plotter.trace_plot(ds, cols)`, `ds.is_stationary(col)` |
| **Trim (single)** | `TrimDataStreamOperation(strategy)(ds, column_name=col)` |
| **Ensemble** | `qnds.Ensemble(trimmed_members)` |
| **Visualize SS** | `plotter.steady_state_automatic_plot(ds, cols)` |
| **Statistics** | `ens.compute_statistics(col, technique=0|1|2)` |
| **Export** | `qnds.Exporter().display_dataframe(r)` / `.display_json(r)` |
| **Strategy swap** | Change the `*TrimStrategy` object; everything else stays the same |
| **Strategy survey** | `build_trim_strategy(method=..., window_size=...)` |

### Key principles

- **Never call `ds.trim()`** — always use the strategy-operation pattern.
- **Drop empty members** after trimming with `[t for t in ... if not t.data.empty]`.
- **Technique 2** is the recommended ensemble estimator for plasma UQ workflows.
- **`build_trim_strategy`** is the preferred entry point for strategy sweeps;
  import individual strategy classes only when you need finer constructor control.

For further reading see the `quends` API reference and the GX tutorial series.